# Semana 12: Beat the baseline

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
df = pd.read_csv(BASE + "churn_telecom.csv")
df.columns = [" ".join(c.split()) for c in df.columns]
y, X = df["Churn"], df.drop(columns="Churn")
from sklearn.model_selection import cross_val_score, StratifiedKFold
def partir(d):
    yy, XX = d["Churn"], d.drop(columns="Churn")
    return train_test_split(XX, yy, test_size=0.25, random_state=42, stratify=yy)

## 1. Revisión de la pista

In [ ]:
print("Filas repetidas:", df.duplicated().sum())
Xtr, Xte, ytr, yte = partir(df)
clave = lambda d: d.astype(str).agg("|".join, axis=1)
print("Clientes de prueba con gemelo en entrenamiento:", clave(Xte).isin(set(clave(Xtr))).sum(), "de", len(Xte))

## 2. La carrera otra vez

In [ ]:
def carrera(d, nombre):
    Xtr, Xte, ytr, yte = partir(d)
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
    ar = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)
    print(nombre, "| prueba", len(yte), "se van", yte.sum(), "| base", round(1 - yte.mean(), 3),
          "| logística", round(lr.score(Xte, yte), 3), round(recall_score(yte, lr.predict(Xte)), 3),
          "| árbol", round(ar.score(Xte, yte), 3), round(recall_score(yte, ar.predict(Xte)), 3), "entrenamiento", round(ar.score(Xtr, ytr), 3))
    return Xtr, Xte, ytr, yte, lr
carrera(df, "Con duplicados")
unico = df.drop_duplicates()
print("Filas únicas:", len(unico))
Xtr, Xte, ytr, yte, lr = carrera(unico, "Sin duplicados")

## 3. Validación cruzada

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
s = cross_val_score(DecisionTreeClassifier(max_depth=10, random_state=42), Xtr, ytr, cv=cv)
print("Profundidad 10:", s.round(3), " media", round(s.mean(), 3), " desviación", round(s.std(), 3))

## 4. Sesgo y varianza

In [ ]:
prof, ent, val = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15], [], []
for d in prof:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    val.append(cross_val_score(t, Xtr, ytr, cv=cv).mean()); ent.append(t.fit(Xtr, ytr).score(Xtr, ytr))
print(pd.DataFrame({"profundidad": prof, "entrenamiento": ent, "validacion": val}).round(3).to_string(index=False))
plt.plot(prof, ent, "o-", label="Entrenamiento"); plt.plot(prof, val, "o-", label="Validación cruzada")
plt.xlabel("Profundidad máxima"); plt.legend(); plt.title("Desde 10, el árbol memoriza más de lo que aprende"); plt.show()

## 5. Competencia
Referencia: la logística sin duplicados con 150 llamadas.

In [ ]:
prob = lr.predict_proba(Xte)[:, 1]
orden = np.argsort(-prob)
print("Encontrados con 150 llamadas:", yte.values[orden[:150]].sum(), "de", yte.sum(), " corte", round(prob[orden[149]], 3))

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.